# Class 09: The Same Word Twice
*Elements of Data Science — Honors*

**First thing:** save this notebook under a new name that includes your team's name.

Part 1 was on paper. This notebook is Part 2. Some code is written for you — every
`...` is yours.

In [ ]:
from datascience import *
import numpy as np
# import for plotting
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
# Fix for datascience plots
import collections as collections
import collections.abc as abc
collections.Iterable = abc.Iterable

In [ ]:
import os
path = 'data/heart.csv' if os.path.exists('data/heart.csv') else 'heart.csv'
heart = Table.read_table(path)
heart.show(5)

Each row is a patient referred for cardiac testing. `target` is 1 if they turned out to
have heart disease and 0 if they did not. `sex` is 1 for men and 0 for women. `cp` is chest
pain type (0–3) and `thalach` is the maximum heart rate they reached under exercise.

---
## A boolean is a probability waiting to be averaged

Comparing a column to a value gives you back an array of `True`/`False` — one answer per
patient, not just one. Keep that in mind for what comes next.

In [ ]:
heart.column('target') == 1

`np.mean` of a `True`/`False` array is the fraction that are `True`. Which is to say:
the probability that a patient picked at random from this table has heart disease.

In [ ]:
np.mean(heart.column('target') == 1)

**2.1** Record that number on your worksheet. That is an *unconditional* probability —
it is about the whole table, with no restriction.

## Conditioning is filtering

In Part 1 you conditioned by covering up the grid and counting only the cells that were
left. `where()` does exactly that to a table: it keeps the rows where a condition is `True`
and throws the rest away.

In [ ]:
women = heart.where('sex', 0)
women.num_rows

In [ ]:
# Of the women, what fraction have heart disease?  P(disease | female)
np.mean(women.column('target') == 1)

**2.2** Now reverse the condition. Restrict to the patients who have heart disease,
and find the fraction of *those* who are women — `P(female | disease)`.

Same two columns. Different question. The numerator is identical and the denominator is
not.

In [ ]:
sick = ...

...

---
## Part 2. Write a Diagnostic Rule

Now the `if` statements. Below is a rule that looks at a patient and returns either
`'flag'` (send them for further testing) or `'clear'`. It is deliberately mediocre.

In [ ]:
def example_rule(cp, thalach, age):
    if cp == 2 or cp == 3:
        return 'flag'
    elif thalach < 140 and age > 55:
        return 'flag'
    else:
        return 'clear'

`apply` runs a function on every row, handing it the columns you name, in order. The
column names must match the function's arguments in order — `cp`, then `thalach`, then
`age`.

In [ ]:
calls = heart.apply(example_rule, 'cp', 'thalach', 'age')
calls

### Grading a rule

There are two ways to grade it, and they are the two conditional probabilities from Part 1.

- **Of the patients who really have heart disease, what fraction did the rule flag?**
  `P(flag | disease)` — this is about catching the sick.
- **Of the patients the rule flagged, what fraction really have heart disease?**
  `P(disease | flag)` — this is about not crying wolf.

The function below computes both. It is written for you so that your twenty minutes goes
into designing rules, not plumbing.

In [ ]:
def score(calls):
    graded = heart.with_column('call', calls)
    flagged = graded.where('call', 'flag')
    sick = graded.where('target', 1)
    print('flagged           ', flagged.num_rows, 'of', graded.num_rows, 'patients')
    print('P(flag | disease) ', np.round(np.mean(sick.column('call') == 'flag'), 3))
    print('P(disease | flag) ', np.round(np.mean(flagged.column('target') == 1), 3))

In [ ]:
score(calls)

**What would a better rule look like?**

Both of `example_rule`'s numbers are only a little above the 51.3% baseline from 2.1 —
not a strong rule. A better rule pushes `P(flag | disease)` *and* `P(disease | flag)` both
higher. That's harder than it sounds: loosen the conditions (flag more people) and you
usually catch more of the sick, so `P(flag | disease)` goes up — but you also sweep in
more people who aren't sick, so `P(disease | flag)` tends to fall. Tighten the conditions
and it often goes the other way. Moving one number up while the other collapses is not an
improvement, even though it can look like one — that's the trap `flag_everyone` (next) is
built to show you.

**2.3** Before you write your own rule, try this one. It is allowed to be stupid.

In [ ]:
def flag_everyone(cp, thalach, age):
    return 'flag'

score(heart.apply(flag_everyone, 'cp', 'thalach', 'age'))

**2.4** Record both numbers for `flag_everyone` on your worksheet, and make sure your
team can say why one of them is perfect.

### Your rule

The fast way in: copy `example_rule` above, give the copy a new name, and change one
thing — a threshold, or swap one condition for a different column. Use `if` / `elif` /
`else`, and make sure the names you pass to `apply` match your arguments, in order.

Useful columns: `age`, `sex`, `cp`, `trestbps` (resting blood pressure), `chol`
(cholesterol), `thalach` (max heart rate), `exang` (exercise-induced angina, 1 = yes),
`oldpeak`, `ca`, `thal`.

In [ ]:
def my_rule(...):
    ...

score(heart.apply(my_rule, ...))

**2.5** Iterate. Change a threshold, run it again, watch both numbers move. Get the
best rule you can and record it on the worksheet — you will have to defend what "best"
means.

In [ ]:
...

---
## Where this goes next

What you just did — try a rule, score it, adjust, try again — is a hand-run version of
what a machine learning algorithm does automatically. A method like the nearest-neighbor
classifier in Lab 10 runs the same kind of search — comparing cases, looking for patterns
that separate one outcome from another — except over far more candidates than a team can
try by hand, and it is normally graded on cases it never saw while learning, not the ones
used to build it. That second part is a real difference from what you did today. Keep it
in mind.